# P13: giải mã theo metric (MBR) cho MER từ vựng mở, pilot zero-shot (Kaggle, 1×T4)
Thiết kế và tiêu chí đặt trước: `explore/ovmer/p13_mbr.py` (docstring). Cần secret **HF_TOKEN** (quyền MER2024). Bật GPU T4, Internet ON.
Kết quả: `/kaggle/working/p13_report.md` (chỉ số tổng hợp) và `p13_out.jsonl` (từ cảm xúc model sinh ra; không có media hay nhãn).

In [ ]:
HOURS = 8.5
KOUT, WORK = '/kaggle/working', '/tmp/p13'
MELD_TAR = 'https://huggingface.co/datasets/declare-lab/MELD/resolve/main/MELD.Raw.tar.gz'

In [ ]:
import os, subprocess
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
os.makedirs(WORK, exist_ok=True)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
sh(f'rm -rf {WORK}/affectgpt && git clone -q --depth 1 https://github.com/zeroQiaoba/AffectGPT {WORK}/affectgpt')
sh(f'rm -rf {WORK}/emos && git clone -q --depth 1 https://github.com/NLP2CT/EmoS {WORK}/emos')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
sh('pip -q install -U huggingface_hub qwen-omni-utils openpyxl', check=False)
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    raise SystemExit('HF_TOKEN secret is not attached to THIS notebook: Add-ons -> Secrets -> tick HF_TOKEN, then Save & Run All again.') from None
os.environ['HF_HOME'] = f'{WORK}/hf'
import torch, transformers; print('cuda:', torch.cuda.is_available(), '| transformers', transformers.__version__)

In [ ]:
from huggingface_hub import hf_hub_download
Z = hf_hub_download(repo_id='MERChallenge/MER2024', repo_type='dataset', filename='video-labeled.zip', local_dir=f'{WORK}/MER2024', token=os.environ['HF_TOKEN'])
print('video-labeled.zip', f'{os.path.getsize(Z) / 1e9:.2f} GB')

In [ ]:
ITEMS = f'{KOUT}/p13_items.jsonl'
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p13_prepare.py', '--affectgpt', f'{WORK}/affectgpt', '--emos', f'{WORK}/emos',
                '--mer24zip', Z, '--meld_tar', MELD_TAR, '--work', f'{WORK}/media', '--out', ITEMS], check=False)

In [ ]:
OUT = f'{KOUT}/p13_out.jsonl'
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p13_mbr.py', 'run', '--items', ITEMS, '--out', OUT, '--hours', str(HOURS)], check=False)

In [ ]:
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p13_mbr.py', 'score', '--items', ITEMS, '--out', OUT,
                '--mapping', f'{WORK}/affectgpt/AffectGPT-RL/emotion_wheel/wheel_mapping.npz', '--report', f'{KOUT}/p13_report.md'], check=False)
print(open(f'{KOUT}/p13_report.md').read())